# UniSchema egress report

Reads **ConstituentEvent** JSON from local egress and summarizes donations by source system.

**Prerequisite:** run `npm run demo` from the repo root (or send a test webhook).

In [ ]:
import json
from pathlib import Path

import pandas as pd

REPO_ROOT = Path.cwd().parents[1] if Path.cwd().name == "downstream" else Path.cwd()
EGRESS_DIR = REPO_ROOT / "data" / "egress"

rows = []
for path in EGRESS_DIR.rglob("*.json"):
    if "manifest" in path.name:
        continue
    data = json.loads(path.read_text(encoding="utf-8"))
    if isinstance(data, dict) and "eventId" in data:
        rows.append(data)

df = pd.DataFrame(rows)
print(f"Loaded {len(df)} ConstituentEvent(s) from {EGRESS_DIR}")
df.head()

In [ ]:
if df.empty:
    print("No events yet — run: npm run demo")
else:
    summary = df.groupby(["sourceSystem", "eventType"], dropna=False).agg(
        events=("eventId", "count"),
        donation_total=("amount", "sum"),
    )
    display(summary)

    if "amount" in df.columns and df["amount"].notna().any():
        donations = df[df["eventType"] == "DONATION"]
        donations.groupby("sourceSystem")["amount"].sum().plot(kind="bar", title="Donations by source system")

## S3 NDJSON batches (production)

Download a batch from S3, then:

```python
lines = open("batch.ndjson").read().splitlines()
df = pd.DataFrame(json.loads(line) for line in lines if line.strip())
```

Or use `read_s3_ndjson_batch.py` from the shell.